In [5]:
# Test the data quality checks
import sys
sys.path.append("../src")

import pandas as pd
from data_quality import run_quality_checks

raw = pd.read_csv("../data/member1_raw_sensor_stream.csv")
clean = run_quality_checks(raw)

print("Rows before:", len(raw))
print("Rows after:", len(clean))
clean["quality_flag"].value_counts()

Rows before: 2959
Rows after: 2941


quality_flag
OK                                          2334
CURRENT_SIGN_REVERSED                        411
TEMP1_SENTINEL_-127                           96
TEMP2_SENTINEL_-127                           35
TEMP1_MISSING_-999                            17
DUPLICATE_RECEIVED                            15
TEMP2_MISSING_-999                            12
TEMP1_SENTINEL_-127|TEMP2_SENTINEL_-127       10
TEMP2_MISSING_-999|CURRENT_SIGN_REVERSED       4
TEMP1_MISSING_-999|CURRENT_SIGN_REVERSED       3
DUPLICATE_RECEIVED|CURRENT_SIGN_REVERSED       2
TEMP1_MISSING_-999|TEMP2_SENTINEL_-127         1
DUPLICATE_RECEIVED|TEMP2_MISSING_-999          1
Name: count, dtype: int64

In [6]:
# compare panel temp before and after cleaning
print("BEFORE cleaning")
print(raw[["panel_temp1_c", "panel_temp2_c"]].mean())
print()
print("AFTER cleaning")
print(clean[["panel_temp1_c", "panel_temp2_c"]].mean())

BEFORE cleaning
panel_temp1_c    23.140294
panel_temp2_c    27.019338
dtype: float64

AFTER cleaning
panel_temp1_c    36.349549
panel_temp2_c    35.888007
dtype: float64


In [7]:
# reload the file because we changed it
import importlib
import data_quality
importlib.reload(data_quality)

clean = data_quality.run_quality_checks(raw)

print("Sessions found:", clean["session_id"].nunique())
print("Reversed current rows:", clean["quality_flag"].str.contains("CURRENT_SIGN_REVERSED").sum())
print("Negative current left:", (clean["current_ma"] < 0).sum())

Sessions found: 50
Reversed current rows: 420
Negative current left: 0


In [8]:
# reload and check the spikes
importlib.reload(data_quality)
clean = data_quality.run_quality_checks(raw)

for flag in ["SPIKE_VOLTAGE", "SPIKE_CURRENT", "SPIKE_IRRADIANCE"]:
    print(flag, clean["quality_flag"].str.contains(flag).sum())

print()
print(clean[["voltage_v", "current_ma", "irradiance_lux", "humidity_pct"]].agg(["min", "max"]))

SPIKE_VOLTAGE 7
SPIKE_CURRENT 9
SPIKE_IRRADIANCE 9

     voltage_v  current_ma  irradiance_lux  humidity_pct
min     -0.021       0.000         663.632        67.847
max      3.338     166.436       98606.204       100.000
